In [ ]:
import yfinance as yf
import pandas
import numpy
import yfinance
import hmmlearn
import sklearn
import matplotlib
import plotly
import ta
import scipy
import statsmodels
import requests
from market_regime_model.hmm_model import evaluate_model, find_optimal_hmm_states

stocks = ['CBA', ' AAPL', 'GOOGL', 'MSFT', 'AMZN'] 
stock = 'CBA'

feature_cols = [
    "Daily Log Return_scaled",
    "20-Day Rolling Volatility_scaled",
    "Distance from 200-Day MA_scaled",
    "Max Drawdown_scaled",
    "Volume Change_scaled",
    "ATR_scaled",
    "RSI_scaled",
    "MACD_scaled",
    "Close_VIX",
]

evaluate_model(stock, feature_cols, )

TypeError: evaluate_model() missing 3 required positional arguments: 'X_train', 'X_test', and 'df_train'

In [ ]:
def get_current_regime(model, X_recent, regime_config): 
    """
    ...
    """




In [4]:
from market_regime_model.hmm_model import main as run_hmm

run_hmm()

FileNotFoundError: [Errno 2] No such file or directory: 'data/VAS_historical_train_data_engineered.csv'

In [84]:
import yfinance as yf 
import pandas as pd
import numpy as np

def engineer_features(stock):
    """
    Fetch historical stock data from Yahoo Finance.

    Args:
        stock (str): The stock ticker symbol.
        period (str): The time period to fetch data for (default is "max").

    Returns:
        stock_info (dict): A dictionary containing the stock information 
        df (pd.DataFrame): A DataFrame containing the historical stock data.
    """

    try: 
        ticker_data = yf.Ticker(stock)
        stock_info = ticker_data.info
        df = ticker_data.history(period="max")
        current_price = df['Close'].iloc[-1]
    except Exception as e:
        print(f"Error fetching data for {stock}: {e}")
        return None, None

    n = int(np.ceil(df.shape[0]*0.05))

    df['Daily Log Return'] = df['Close'].pct_change().apply(lambda x: np.log(1 + x))
    df['20-day Rolling Vol Annualised'] = df['Daily Log Return'].rolling(window=20).std() * (252 ** 0.5)
    df['200-Day Moving Average'] = df['Close'].rolling(window=200).mean()
    df['Distance from 200-Day MA'] = df['Close'] - df['200-Day Moving Average']
    rolling_peak = df['Close'].rolling(window=20, min_periods=1).max()
    df['Max Drawdown'] = (df['Close'] - rolling_peak) / rolling_peak
    df['Volume Change'] = df['Volume'].pct_change()

    df['Historical Loss'] = current_price*df['Daily Log Return'] # loss based on daily log returns, scaled to latest price

    normal_VaR = current_price*(df['Daily Log Return'].mean() - 1.64485*(df['Daily Log Return'].std())) # VaR assuming normality of stock returns 
    stock_info['Normal 95% CVaR'] = -df['Historical Loss'][df['Historical Loss'] < normal_VaR].mean()
    stock_info['Historical 95% CVaR'] = -df['Historical Loss'].nsmallest(n).mean() # takes the highest 5% magnitude losses and averages to find CVaR

    return stock_info, df

def get_current_regime(model, X_recent, regime_config): 
    ...

def get_conditional_returns():
    ...

def main():
    # model = run_hmm()
    stock_features = {}
    stocks = ['^VIX']
    weights = [1]
    if (len(stocks) != len(weights)): 
        print("Number of stocks and weights should be equal")
        return -1
    adjusted_returns = pd.DataFrame()
    for stock in stocks: 
        stock_feature, data = engineer_features(stock)
        stock_features[stock] = stock_feature
        # data['Daily Log Return'] = 
        # adjusted_returns['']
        print(stock_features[stock]['Historical 95% CVaR'])
        print(stock_features[stock]['Normal 95% CVaR'])

if __name__ == "__main__":
    main()



2.050329974507748
2.266674235128268


In [ ]:
import numpy as np
ticker = yf.Ticker('AAPL')
df = ticker.history(period="max")
current_price = df['Close'].iloc[-1]
df['Daily Log Return'] = df['Close'].pct_change().apply(lambda x: np.log(1 + x))
df['Historical Loss'] = current_price*df['Daily Log Return'] # loss based on daily log returns, scaled to latest price

normal_VaR = current_price*(df['Daily Log Return'].mean() - 1.64485*(df['Daily Log Return'].std()))
normal_CVaR = df['Historical Loss'][df['Historical Loss'] < normal_VaR]

AttributeError: 'numpy.float64' object has no attribute 'as_type'

In [60]:
443/11518

0.038461538461538464